# Adaptive Product Search Agent — ESCI QLoRA training and serving
This notebook fine-tunes **Qwen2.5-0.5B-Instruct** to classify Amazon Shopping Queries ESCI query-product pairs as Exact, Substitute, Complement, or Irrelevant. It measures the frozen base model first, trains a LoRA adapter, measures the adapter on the same held-out set, saves both predictions, and serves the adapter to the local application. No metric in this notebook is hard-coded.

Use a Colab T4 GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!pip -q install 'transformers>=4.52,<5' 'peft>=0.15' 'trl>=0.18' 'datasets>=3.6' 'accelerate>=1.7' 'bitsandbytes>=0.45' 'scikit-learn>=1.5' pandas pyarrow fastapi uvicorn pyngrok nest_asyncio
!rm -rf /content/esci-data
!git clone --depth 1 https://github.com/amazon-science/esci-data.git /content/esci-data
!cd /content/esci-data && git lfs pull
import json, re, random, threading, time
from pathlib import Path
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer
assert torch.cuda.is_available(), 'Enable a T4 GPU before continuing.'
print(torch.cuda.get_device_name(0))

In [ ]:
# Load the official ESCI parquet files and create leakage-safe query-level splits.
root = Path('/content/esci-data')
examples_path = next(p for p in root.rglob('*.parquet') if 'examples' in p.name.lower())
products_path = next(p for p in root.rglob('*.parquet') if 'products' in p.name.lower())
examples = pd.read_parquet(examples_path)
products = pd.read_parquet(products_path)
examples = examples[(examples.product_locale == 'us') & (examples.small_version == 1)].copy()
products = products[products.product_locale == 'us'].copy()
frame = examples.merge(products, on=['product_locale', 'product_id'], how='inner', validate='many_to_one')
official_test = frame[frame['split'].astype(str).str.lower() == 'test'].copy()
official_train = frame[frame['split'].astype(str).str.lower() != 'test'].copy()
query_ids = official_train.query_id.drop_duplicates().sample(frac=1, random_state=42)
validation_ids = set(query_ids.head(max(1, len(query_ids)//10)))
validation_frame = official_train[official_train.query_id.isin(validation_ids)]
train_frame = official_train[~official_train.query_id.isin(validation_ids)]
def balanced_sample(df, total, seed=42):
    per_label = max(1, total // 4)
    parts = [g.sample(min(len(g), per_label), random_state=seed) for _, g in df.groupby('esci_label')]
    return pd.concat(parts).sample(frac=1, random_state=seed).head(total).reset_index(drop=True)
train_frame = balanced_sample(train_frame, 24000)
validation_frame = balanced_sample(validation_frame, 2000)
test_frame = balanced_sample(official_test, 1000)
# Optional matched variant from scripts/prepare_hard_negatives.py. Upload the directory
# to /content/esci-training-variants; default training remains the documented baseline.
TRAINING_VARIANT = 'baseline'  # change only for the matched hard_negative ablation
variant_dir = Path('/content/esci-training-variants')
variant_path = variant_dir / f'{TRAINING_VARIANT}.jsonl'
training_variant_metadata = {'name': 'balanced_random_baseline', 'source': 'not supplied'}
if variant_path.exists():
    variant_rows = [json.loads(line) for line in variant_path.read_text().splitlines() if line.strip()]
    train_frame = pd.DataFrame([{'query': row['query'], 'esci_label': row['label'], 'product_title': row['product'].get('title', ''), 'product_brand': row['product'].get('brand', ''), 'product_color': row['product'].get('colour', ''), 'product_bullet_point': row['product'].get('bullet_points', ''), 'product_description': row['product'].get('description', '')} for row in variant_rows])
    training_variant_metadata = json.loads((variant_dir / 'manifest.json').read_text()) | {'name': TRAINING_VARIANT, 'source': str(variant_path)}
    assert not {str(row['query_id']) for row in variant_rows} & set(test_frame.query_id.astype(str)), 'Variant leaked an official test query'
print('train/validation/test:', len(train_frame), len(validation_frame), len(test_frame))
print(train_frame.esci_label.value_counts().sort_index())

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER_DIR = '/content/esci-qwen-lora'
LABEL_HELP = 'E=exact match, S=usable substitute, C=complement/accessory, I=irrelevant'
def clean(value):
    return '' if pd.isna(value) else str(value)
def classification_prompt(row):
    return f'''Classify the relationship between a shopping query and product.
{LABEL_HELP}.
Return only JSON: {{"label":"E|S|C|I"}}
Query: {clean(row['query'])}
Product title: {clean(row['product_title'])}
Brand: {clean(row['product_brand'])}
Color: {clean(row['product_color'])}
Bullet points: {clean(row['product_bullet_point'])[:900]}
Description: {clean(row['product_description'])[:900]}
JSON:'''.strip()
def training_text(row):
    return classification_prompt(row) + json.dumps({'label': str(row['esci_label'])}, separators=(',', ':'))
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant, device_map='auto')
def predict_labels(rows, limit=None):
    predictions = []
    subset = rows if limit is None else rows.head(limit)
    model.eval()
    for _, row in subset.iterrows():
        inputs = tokenizer(classification_prompt(row), return_tensors='pt', truncation=True, max_length=1536).to(model.device)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        match = re.search(r'"label"\s*:\s*"([ESCI])"', text)
        predictions.append(match.group(1) if match else 'I')
    return predictions
EVAL_ROWS = test_frame.head(500).copy()
base_predictions = predict_labels(EVAL_ROWS)
base_metrics = {'accuracy': accuracy_score(EVAL_ROWS.esci_label, base_predictions), 'macro_f1': f1_score(EVAL_ROWS.esci_label, base_predictions, average='macro', labels=list('ESCI'), zero_division=0)}
print('Frozen base metrics:', json.dumps(base_metrics, indent=2))

In [ ]:
# Supervised fine-tuning: only LoRA matrices are trainable; base weights remain frozen.
train_dataset = Dataset.from_dict({'text': [training_text(row) for _, row in train_frame.iterrows()]})
validation_dataset = Dataset.from_dict({'text': [training_text(row) for _, row in validation_frame.iterrows()]})
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj'])
args = TrainingArguments(output_dir='/content/esci-checkpoints', num_train_epochs=2, per_device_train_batch_size=4, gradient_accumulation_steps=4, learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type='cosine', logging_steps=25, save_strategy='epoch', eval_strategy='epoch', fp16=True, optim='paged_adamw_8bit', report_to='none', seed=42)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=validation_dataset, peft_config=lora, processing_class=tokenizer)
training_started = time.perf_counter()
train_result = trainer.train()
training_seconds = round(time.perf_counter() - training_started, 2)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
model = trainer.model
adapter_predictions = predict_labels(EVAL_ROWS)
adapter_metrics = {'accuracy': accuracy_score(EVAL_ROWS.esci_label, adapter_predictions), 'macro_f1': f1_score(EVAL_ROWS.esci_label, adapter_predictions, average='macro', labels=list('ESCI'), zero_division=0)}
comparison = {'protocol': {'dataset':'Amazon Shopping Queries ESCI','held_out_examples':len(EVAL_ROWS),'seed':42, 'training_variant': training_variant_metadata}, 'frozen_base':base_metrics, 'fine_tuned_lora':adapter_metrics, 'delta':{k:adapter_metrics[k]-base_metrics[k] for k in base_metrics}}
Path('/content/esci-comparison.json').write_text(json.dumps(comparison, indent=2))
Path('/content/esci-training-metadata.json').write_text(json.dumps({'training_seconds': training_seconds, 'train_examples': len(train_frame), 'validation_examples': len(validation_frame), 'base_model': BASE_MODEL, 'adapter': {'rank': 16, 'alpha': 32, 'target_modules': ['q_proj','k_proj','v_proj','o_proj']}, 'variant': training_variant_metadata}, indent=2))
with open('/content/esci-predictions.jsonl','w') as handle:
    for (_, row), base, tuned in zip(EVAL_ROWS.iterrows(), base_predictions, adapter_predictions):
        handle.write(json.dumps({'example_id':str(row.example_id),'query_id':str(row.query_id),'gold':str(row.esci_label),'base_prediction':base,'prediction':tuned})+'\n')
print(json.dumps(comparison, indent=2))

In [ ]:
# Persist the trained adapter and honest before/after evidence to Google Drive.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/adaptive-product-search'
!cp -r /content/esci-qwen-lora '/content/drive/MyDrive/adaptive-product-search/'
!cp /content/esci-comparison.json /content/esci-predictions.jsonl /content/esci-training-metadata.json '/content/drive/MyDrive/adaptive-product-search/'
print('Saved under MyDrive/adaptive-product-search/')

## Serve this exact fine-tuned adapter
Add two Colab secrets (key icon on the left): `NGROK_AUTHTOKEN` from ngrok and a private random `MODEL_API_KEY` of your choice. The cell below exposes the adapter to your local FastAPI app. Keep the cell and Colab runtime running while using the website.

In [ ]:
from typing import Any
import nest_asyncio, uvicorn
from fastapi import FastAPI, Header, HTTPException, Depends
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
MODEL_API_KEY = userdata.get('MODEL_API_KEY')
assert NGROK_TOKEN and MODEL_API_KEY, 'Create both Colab secrets described above.'
ngrok.set_auth_token(NGROK_TOKEN)
serve_app = FastAPI(title='ESCI LoRA inference')
generation_lock = threading.Lock()
def authorize(authorization: str | None = Header(default=None)):
    if authorization != f'Bearer {MODEL_API_KEY}': raise HTTPException(401, 'Invalid model API key')
def generate_json(prompt, max_tokens=256):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
    with generation_lock, torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    match = re.search(r'\{.*\}', text, re.S)
    if not match: raise HTTPException(502, f'Model returned invalid JSON: {text[:200]}')
    try: return json.loads(match.group(0))
    except json.JSONDecodeError as exc: raise HTTPException(502, f'Model returned invalid JSON: {exc}')
@serve_app.get('/health')
def model_health(_: Any = Depends(authorize)):
    return {'status':'ok','model_kind':'fine_tuned_adapter','adapter_loaded':True,'model_version':'qwen2.5-0.5b-esci-lora-r16','base_model':BASE_MODEL}
@serve_app.post('/parse-intent')
def parse_intent(body: dict, _: Any = Depends(authorize)):
    prompt = f'''You are the intent component of a product-search agent running with an ESCI-fine-tuned adapter. Return only JSON with category, required_attributes, preferred_attributes, excluded_attributes, max_price, min_price, referenced_result, clarification_required, search_strategy. Current constraints: {json.dumps(body.get('constraints',{}))}. Recent conversation: {json.dumps(body.get('history',[]))}. New request: {body['message']}'''
    return generate_json(prompt)
@serve_app.post('/rerank')
def rerank(body: dict, _: Any = Depends(authorize)):
    predictions=[]
    for product in body['products']:
        row={'query':body['query'],'product_title':product.get('title',''),'product_brand':product.get('brand',''),'product_color':product.get('colour',''),'product_bullet_point':product.get('bullet_points',''),'product_description':product.get('description','')}
        result=generate_json(classification_prompt(row), 32)
        label=result.get('label','I')
        if label not in 'ESCI': label='I'
        # No confidence is returned until a validation-calibrated estimate is implemented.
        predictions.append({'product_id':product['id'],'label':label,'rationale':f'Adapter ESCI prediction: {label}'})
    return {'predictions':predictions}
@serve_app.post('/answer')
def answer(body: dict, _: Any = Depends(authorize)):
    context=[{'rank':i+1,'product_id':r['product']['id'],'title':r['product']['title'],'description':r['product']['description'],'price':r['product'].get('price'),'label':r['relevance_label']} for i,r in enumerate(body['results'])]
    prompt=f'''Answer the shopping request using only this ranked context. Cite products using their product_id. Return only JSON: {{"answer":"...","citations":[{{"product_id":"...","rank":1}}]}}. Request: {body['query']}. Context: {json.dumps(context)}'''
    return generate_json(prompt, 220)
nest_asyncio.apply()
public_url = ngrok.connect(8000, bind_tls=True).public_url
print('FINETUNED_MODEL_URL=' + public_url)
print('Set FINETUNED_MODEL_API_KEY locally to the MODEL_API_KEY secret value.')
threading.Thread(target=lambda: uvicorn.run(serve_app, host='0.0.0.0', port=8000), daemon=True).start()